In [6]:
import pandas as pd
import glob
import os
import numpy as np
import shutil
import ast
from Granger_dictionaries import *

In [7]:
dir = "..\\data\\granger\\field\\MID\\2025"

idx = 3  # Change this index to select a different file
db = "MID"
collection_quarter = "4Q25"

task_code = f"{db}_{collection_quarter}"
sampling_company_code = "STRATA"
# Get all Excel file paths in the directory
excel_paths = glob.glob(os.path.join(dir, "*.xlsx"))
print(excel_paths)



EDD_NAME = excel_paths[idx].split("\\")[-1]
EDD_TO_PROCESS = pd.read_excel(f'{dir}\\{EDD_NAME}', header=None)
# Find coordinates of "Project No.:"
row_idx, col_pos = np.where(EDD_TO_PROCESS == "Project No.:")
# project_no = EDD_TO_PROCESS.iat[row_idx[0], col_pos[0] + 1]
# print(f"Project No.: {project_no}")
print(EDD_NAME)
EDD_TO_PROCESS.shape

['..\\data\\granger\\field\\MID\\2025\\-1160 Q1 2025 Params_Table 2.xlsx', '..\\data\\granger\\field\\MID\\2025\\-1160 Q2 2025 Params_Table 2.xlsx', '..\\data\\granger\\field\\MID\\2025\\-1160 Q3 2025 Params_Table 2.xlsx', '..\\data\\granger\\field\\MID\\2025\\-1160 Q4 2025 Params_Table 2.xlsx']
-1160 Q4 2025 Params_Table 2.xlsx


(20, 6)

In [8]:
cas_dict = {"pH, Field": "pH-F", "Temperature, Field": "Temp-F", "Specific Conductivity, Field": "SpCond-F"}
unit_dict = {"pH-F":"su", "Temp-F": "Deg C", "SpCond-F": "mS/cm"}

# Find first row where column 0 starts with "MW-"
start_row = EDD_TO_PROCESS[0].astype(str).str.match(r"MW-").idxmax()

EDD = EDD_TO_PROCESS.copy()
EDD = (
    EDD.loc[start_row:, [0, 1, 2, 3, 4]]
    .reset_index(drop=True)
    .set_axis(
        [
            "sys_loc_code",
            "result_date",
            "pH, Field",
            "Temperature, Field",
            "Specific Conductivity, Field"
        ],
        axis=1
    )
)
# Drop row if no date:
EDD = EDD.dropna(subset=["result_date"])
EDD["sys_loc_code"] = EDD["sys_loc_code"].astype(str).str.strip()

pattern = r"mw-|leachate|sw-|p-|pw|surface|underdrain|cell|lc"
EDD = EDD[
    EDD["sys_loc_code"]
    .str.lower()
    .str.contains(pattern, na=False)
]

# gets the result value from each field
cols = [
    "pH, Field",
    "Temperature, Field",
    "Specific Conductivity, Field"
]

for col in cols:
    EDD[col] = EDD[col].apply(lambda x: list(str(x).split(","))[-1])

# Unpivot (Melt), the field measurements into the format we need
EDD = EDD.melt(
    id_vars=["sys_loc_code", "result_date"],
    value_vars=[
        "pH, Field",
        "Temperature, Field",
        "Specific Conductivity, Field"
    ],
    var_name="field_parameter_name",
    value_name="result_value"
).sort_values(by=["sys_loc_code", "result_date"])

EDD['sample_name'] = EDD.sys_loc_code
EDD['sys_loc_code'] = EDD['sys_loc_code'].map(sample_id_dict).fillna(EDD['sys_loc_code'])  
EDD['task_code'] = task_code
EDD['sampling_company_code'] = sampling_company_code
EDD['reportable_result'] = 'YES'
EDD['collection_quarter'] = collection_quarter
# Where 'Result' is empty, set 'detect_flag' to "N" otherwise set it to "Y"
EDD['detect_flag'] = np.where(EDD['result_value'] == '', "N", "Y")
EDD['field_parameter_code'] = EDD['field_parameter_name'].map(cas_dict)
EDD['result_unit'] = EDD['field_parameter_code'].map(unit_dict)
EDD['#data_provider'] = sampling_company_code

EDD['value_type'] = "ACTUAL"
EDD['analytic_method'] = "Field"
EDD['sample_matrix_code'] = EDD['sys_loc_code'].apply(assign_matrix)

dates = pd.to_datetime(EDD["result_date"], errors="coerce")
EDD["sys_sample_code"] = np.where(
    dates.notna(),
    EDD["sample_name"].astype(str) + "_" + dates.dt.strftime("%Y%m%d"),
    np.nan
)

In [9]:
# 1. path to template
template_path = r"..\EDDTemplate\Field EDD Template.xlsx"
template_df = pd.read_excel(template_path, sheet_name="FieldResults_v1")

EDD = EDD.reindex(columns=template_df.columns)
expected_columns = ["#data_provider", "sys_loc_code", "sys_sample_code", "field_parameter_code", "field_parameter_name", "result_date", "result_time", "start_depth", "end_depth", "depth_unit", "result_value", "result_unit", "quantitation_limit", "task_code", "sample_matrix_code", "qualifier", "sampling_company_code", "sampling_reason", "sample_method", "reportable_result", "value_type", "remark", "detect_flag", "collection_quarter", "sample_name", "lab_name_code", "analytic_method", "validator_qualifiers", "fraction", "custom_field_4", "TRC_Project_Number", "sampler"]

try:
    assert EDD.columns.tolist() == expected_columns
    print("Column names match the expected list.")
except AssertionError:
    print("Column names do not match the expected list.")
    # The EDD to process has extra columns that are not in the expected list. Print the extra columns.
    print("Extra columns in EDD to process:", set(template_df.columns.tolist()) - set(expected_columns
    ))

# 2. Make an exact copy of the template file
# Ouptut path for the new EDD file to be created
# Make a unique name for the output file by including the original file name and a timestamp
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
output_path = f"{dir}\\{EDD_NAME.split('.')[0]}_FIELD_EDD_{timestamp}.xlsx"
shutil.copy(template_path, output_path)

# 3. Write your dataframe into the copied template
# (Using 'a' mode allows you to append/write data to an existing sheet)
with pd.ExcelWriter(
    output_path, engine="openpyxl", mode="a", if_sheet_exists="overlay"
) as writer:
    EDD.to_excel(
        writer,
        sheet_name="FieldResults_v1",  # <-- Change this to the exact name of the sheet in your template
        index=False,
        header=False,  # <-- Set to False if your template already has the headers typed out
        startrow=2,  # <-- Starts writing on row 2 (0-indexed), assuming row 1 has your headers
    )

Column names do not match the expected list.
Extra columns in EDD to process: set()


c:\ageglio-1\EDD Data Processor\dataprocessor\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\ageglio-1\EDD Data Processor\dataprocessor\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [10]:
EDD

,#data_provider,sys_loc_code,sys_sample_code,field_parameter_code,field_parameter_name,start_depth,end_depth,depth_unit,result_date,result_time,...,detect_flag,collection_quarter,sample_name,lab_name_code,analytic_method,validator_qualifiers,fraction,custom_field_4,TRC_Project_Number,sampler
0,STRATA,MW-19r,MW-19r_20251009,pH-F,"pH, Field",NaN,NaN,NaN,2025-10-09,NaN,...,Y,4Q25,MW-19r,NaN,Field,NaN,NaN,NaN,NaN,NaN
11,STRATA,MW-19r,MW-19r_20251009,Temp-F,"Temperature, Field",NaN,NaN,NaN,2025-10-09,NaN,...,Y,4Q25,MW-19r,NaN,Field,NaN,NaN,NaN,NaN,NaN
22,STRATA,MW-19r,MW-19r_20251009,SpCond-F,"Specific Conductivity, Field",NaN,NaN,NaN,2025-10-09,NaN,...,Y,4Q25,MW-19r,NaN,Field,NaN,NaN,NaN,NaN,NaN
1,STRATA,MW-21sr2,MW-21sr2_20251008,pH-F,"pH, Field",NaN,NaN,NaN,2025-10-08,NaN,...,Y,4Q25,MW-21sr2,NaN,Field,NaN,NaN,NaN,NaN,NaN
12,STRATA,MW-21sr2,MW-21sr2_20251008,Temp-F,"Temperature, Field",NaN,NaN,NaN,2025-10-08,NaN,...,Y,4Q25,MW-21sr2,NaN,Field,NaN,NaN,NaN,NaN,NaN
23,STRATA,MW-21sr2,MW-21sr2_20251008,SpCond-F,"Specific Conductivity, Field",NaN,NaN,NaN,2025-10-08,NaN,...,Y,4Q25,MW-21sr2,NaN,Field,NaN,NaN,NaN,NaN,NaN
2,STRATA,MW-24dr2,MW-24dr2_20251008,pH-F,"pH, Field",NaN,NaN,NaN,2025-10-08,NaN,...,Y,4Q25,MW-24dr2,NaN,Field,NaN,NaN,NaN,NaN,NaN
13,STRATA,MW-24dr2,MW-24dr2_20251008,Temp-F,"Temperature, Field",NaN,NaN,NaN,2025-10-08,NaN,...,Y,4Q25,MW-24dr2,NaN,Field,NaN,NaN,NaN,NaN,NaN
24,STRATA,MW-24dr2,MW-24dr2_20251008,SpCond-F,"Specific Conductivity, Field",NaN,NaN,NaN,2025-10-08,NaN,...,Y,4Q25,MW-24dr2,NaN,Field,NaN,NaN,NaN,NaN,NaN
3,STRATA,MW-25r2,MW-25r2_20251009,pH-F,"pH, Field",NaN,NaN,NaN,2025-10-09,NaN,...,Y,4Q25,MW-25r2,NaN,Field,NaN,NaN,NaN,NaN,NaN
